# Phosphogypsum Treatment Optimization

So the idea here is pretty simple. Phosphogypsum is a byproduct of phosphoric acid production, and before we can just dump it, we need to treat it — basically wash it and neutralize it with lime milk. Doing that properly is tricky because the right amount of lime and the right washing time depend heavily on what's actually in the sample (acidity, fluorides, heavy metals, all that stuff).

This notebook builds a small ML system that takes the chemical composition of a phosphogypsum sample and predicts the optimal treatment parameters. We train XGBoost and Random Forest side by side, keep whichever does better for each target, and save everything so it can be reused later without retraining.

## 1. Setup

Just the usual imports. Nothing fancy.

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

os.makedirs("models", exist_ok=True)

ModuleNotFoundError: No module named 'joblib'

## 2. Generating the dataset

Since we don't have real experimental data yet, we generate a synthetic one based on typical phosphogypsum chemistry reported in the literature. The ranges for each element are realistic (P₂O₅ around 0.5–5%, CaO ~30%, SO₃ ~42%, fluorides a few tenths of a percent, heavy metals in ppm, etc.).

The targets (lime dose, washing time, recovery, cost, final pH) are computed from physically motivated formulas — more acid means more lime, more impurities means longer washing, that kind of thing — with some noise added so it looks like real measurements.

When you have real lab data later, you just drop it in as a CSV with the same columns and rerun. Everything else stays the same.

In [ ]:
def generate_dataset(n_samples=5000, seed=42):
    rng = np.random.default_rng(seed)

    P2O5 = rng.uniform(0.5, 5.0, n_samples)
    CaO = rng.uniform(28.0, 33.0, n_samples)
    SO3 = rng.uniform(40.0, 46.0, n_samples)
    F = rng.uniform(0.2, 2.0, n_samples)
    SiO2 = rng.uniform(0.5, 5.0, n_samples)
    Fe2O3 = rng.uniform(0.05, 0.8, n_samples)
    Al2O3 = rng.uniform(0.05, 0.5, n_samples)
    MgO = rng.uniform(0.01, 0.3, n_samples)
    Na2O = rng.uniform(0.05, 0.4, n_samples)
    K2O = rng.uniform(0.01, 0.2, n_samples)
    Cd = rng.uniform(0.5, 15.0, n_samples)
    Pb = rng.uniform(1.0, 25.0, n_samples)
    Zn = rng.uniform(10.0, 200.0, n_samples)
    As_ = rng.uniform(0.5, 20.0, n_samples)
    Ra226 = rng.uniform(100.0, 1000.0, n_samples)
    moisture = rng.uniform(10.0, 25.0, n_samples)
    pH = rng.uniform(2.5, 5.5, n_samples)
    temperature = rng.uniform(15.0, 45.0, n_samples)

    acidity_factor = (5.5 - pH) / 3.0
    lime_milk = (
        15.0 + 8.0 * P2O5 + 12.0 * F + 25.0 * acidity_factor
        + 0.3 * SiO2 + 0.05 * (temperature - 25)
        + rng.normal(0, 2.0, n_samples)
    )
    lime_milk = np.clip(lime_milk, 5.0, 120.0)

    impurity_load = SiO2 + Fe2O3 + Al2O3 + (Cd + Pb + As_) / 50.0
    washing_time = (
        20.0 + 4.0 * impurity_load + 0.5 * moisture + 3.0 * F + 2.0 * P2O5
        - 0.1 * temperature + rng.normal(0, 3.0, n_samples)
    )
    washing_time = np.clip(washing_time, 10.0, 120.0)

    recovery = (
        60.0 + 5.0 * P2O5 - 2.0 * F - 0.02 * Ra226 / 10
        + 0.1 * washing_time + 0.05 * lime_milk + 0.2 * (temperature - 20)
        + rng.normal(0, 3.0, n_samples)
    )
    recovery = np.clip(recovery, 40.0, 98.0)

    cost = (
        5.0 + lime_milk * 0.15 + washing_time * 0.08
        + 0.02 * (Cd + Pb + As_) + 0.005 * Ra226
        + rng.normal(0, 0.5, n_samples)
    )
    cost = np.clip(cost, 5.0, 60.0)

    final_pH = 6.5 + (lime_milk - 30) * 0.03 + rng.normal(0, 0.2, n_samples)
    final_pH = np.clip(final_pH, 5.5, 9.0)

    return pd.DataFrame({
        "P2O5_percent": P2O5, "CaO_percent": CaO, "SO3_percent": SO3,
        "F_percent": F, "SiO2_percent": SiO2, "Fe2O3_percent": Fe2O3,
        "Al2O3_percent": Al2O3, "MgO_percent": MgO, "Na2O_percent": Na2O,
        "K2O_percent": K2O, "Cd_ppm": Cd, "Pb_ppm": Pb, "Zn_ppm": Zn,
        "As_ppm": As_, "Ra226_Bq_per_kg": Ra226,
        "moisture_percent": moisture, "pH_initial": pH, "temperature_C": temperature,
        "lime_milk_kg_per_ton": lime_milk, "washing_time_min": washing_time,
        "P2O5_recovery_percent": recovery, "treatment_cost_USD_per_ton": cost,
        "final_pH": final_pH,
    })

df = generate_dataset(5000)
df.to_csv("phosphogypsum_dataset.csv", index=False)
df.head()

Quick look at the distributions to make sure nothing looks weird.

In [ ]:
df.describe().round(2)

## 3. Splitting inputs and targets

18 input features (the composition + conditions), 5 targets to predict. We scale the inputs with a standard scaler — it doesn't really matter for tree models, but it makes the scaler available if we later swap in something like a neural net or SVM.

In [ ]:
INPUT_FEATURES = [
    "P2O5_percent", "CaO_percent", "SO3_percent", "F_percent",
    "SiO2_percent", "Fe2O3_percent", "Al2O3_percent", "MgO_percent",
    "Na2O_percent", "K2O_percent", "Cd_ppm", "Pb_ppm", "Zn_ppm", "As_ppm",
    "Ra226_Bq_per_kg", "moisture_percent", "pH_initial", "temperature_C",
]

TARGET_FEATURES = [
    "lime_milk_kg_per_ton", "washing_time_min",
    "P2O5_recovery_percent", "treatment_cost_USD_per_ton", "final_pH",
]

X = df[INPUT_FEATURES].values
y = df[TARGET_FEATURES].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

joblib.dump(scaler, "models/scaler.pkl")
print(X_train_s.shape, X_test_s.shape)

## 4. Training the models

For every one of the 5 targets we train both XGBoost and Random Forest, then we pick whichever gives the higher R² on the held-out test set. The winner gets saved to disk. That way the final system is basically a small ensemble of best-performing models, one per output.

I kept the hyperparameters reasonable — not tuned to death, just solid defaults. If you want to squeeze more performance later you can throw an Optuna or GridSearch at this.

In [ ]:
def train_one_target(X_tr, X_te, y_tr, y_te, target_name):
    xgb = XGBRegressor(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        random_state=42, n_jobs=-1, verbosity=0,
    )
    xgb.fit(X_tr, y_tr)
    xgb_pred = xgb.predict(X_te)

    rf = RandomForestRegressor(
        n_estimators=300, max_depth=18, min_samples_split=4,
        random_state=42, n_jobs=-1,
    )
    rf.fit(X_tr, y_tr)
    rf_pred = rf.predict(X_te)

    scores = {
        "xgboost": {
            "model": xgb,
            "r2": r2_score(y_te, xgb_pred),
            "mae": mean_absolute_error(y_te, xgb_pred),
            "rmse": float(np.sqrt(mean_squared_error(y_te, xgb_pred))),
        },
        "random_forest": {
            "model": rf,
            "r2": r2_score(y_te, rf_pred),
            "mae": mean_absolute_error(y_te, rf_pred),
            "rmse": float(np.sqrt(mean_squared_error(y_te, rf_pred))),
        },
    }
    best = max(scores, key=lambda k: scores[k]["r2"])
    print(f"[{target_name}]")
    for name, s in scores.items():
        mark = "  <-- keeping" if name == best else ""
        print(f"  {name:15s}  R2={s['r2']:.4f}  MAE={s['mae']:.3f}  RMSE={s['rmse']:.3f}{mark}")
    return best, scores

best_algo = {}
all_metrics = {}
for i, target in enumerate(TARGET_FEATURES):
    best, scores = train_one_target(X_train_s, X_test_s, y_train[:, i], y_test[:, i], target)
    best_algo[target] = best
    joblib.dump(scores[best]["model"], f"models/{target}__{best}.pkl")
    all_metrics[target] = {k: {kk: vv for kk, vv in v.items() if kk != "model"} for k, v in scores.items()}
    print()

## 5. Saving metadata

A tiny JSON file that tells the pipeline which algorithm won for each target and what the input columns are. This is what `pipeline.py` reads at load time — no hardcoded paths, no guessing.

In [ ]:
meta = {
    "input_features": INPUT_FEATURES,
    "target_features": TARGET_FEATURES,
    "best_algorithm_per_target": best_algo,
    "metrics": all_metrics,
}
with open("models/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

sorted(os.listdir("models"))

## 6. Quick sanity check

Let's throw one sample at the saved models and see if the output makes sense. For a pretty acidic sample with lots of fluoride we should get a high lime dose, longer washing, and a reasonable final pH close to neutral.

In [ ]:
sample = {
    "P2O5_percent": 3.2, "CaO_percent": 29.5, "SO3_percent": 43.5,
    "F_percent": 1.6, "SiO2_percent": 2.8, "Fe2O3_percent": 0.4,
    "Al2O3_percent": 0.25, "MgO_percent": 0.12, "Na2O_percent": 0.18,
    "K2O_percent": 0.09, "Cd_ppm": 8.0, "Pb_ppm": 15.0, "Zn_ppm": 100.0,
    "As_ppm": 10.0, "Ra226_Bq_per_kg": 600.0, "moisture_percent": 20.0,
    "pH_initial": 3.0, "temperature_C": 22.0,
}

x = scaler.transform(pd.DataFrame([sample])[INPUT_FEATURES].values)
preds = {}
for target, algo in best_algo.items():
    model = joblib.load(f"models/{target}__{algo}.pkl")
    preds[target] = float(model.predict(x)[0])
preds

## 7. That's it

All the trained models, the scaler, and the metadata are now sitting in the `models/` folder. From here on you don't need this notebook anymore — the `pipeline.py` file takes care of loading everything and predicting on new samples. Check the docs file for usage examples.